In [2]:
import os

In [3]:
pwd

'd:\\projects\\AI-Tex-Summraiser\\research'

In [4]:
os.chdir("../")

In [5]:
pwd

'd:\\projects\\AI-Tex-Summraiser'

In [6]:
from dataclasses import dataclass 
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationconfig:
    root_dir: Path
    data_path:Path
    model_path:Path
    tokenizer_path: Path
    metric_file_name:Path

In [7]:
from ai_text_summariser.constants import * 
from ai_text_summariser.utils.common import read_yaml, create_directories
from ai_text_summariser.logging import logger


In [8]:
from pathlib import Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAM_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationconfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationconfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [9]:
pip install evaluate


Note: you may need to restart the kernel to use updated packages.


In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
from tqdm import tqdm

c:\Users\Vandan Prajapati\anaconda3\envs\aitext310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-06-11 11:25:08,778: INFO: config: PyTorch version 2.7.0 available.]


In [11]:
from pathlib import Path

print(Path("artifacts/model_trainer/tokenizer").absolute().exists())


True


In [12]:
import evaluate
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import pandas as pd
from tqdm import tqdm

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationconfig):
        self.config = config

    def generate_batch_sized_chunks(self, elements, batch_size):
        for i in range(0, len(elements), batch_size):
            yield elements[i: i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer,
                                    batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                    column_text="article", column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
            inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                               padding="max_length", return_tensors="pt")

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8, num_beams=8, max_length=128
            )

            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        return metric.compute()

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer_path = Path(self.config.tokenizer_path).resolve()
        model_path = Path(self.config.model_path).resolve()

        if not tokenizer_path.exists():
            raise FileNotFoundError(f"Tokenizer path does not exist: {tokenizer_path}")
        if not model_path.exists():
            raise FileNotFoundError(f"Model path does not exist: {model_path}")

        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path.as_posix(), local_files_only=True)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path.as_posix(), local_files_only=True).to(device)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt["test"][:10], rouge_metric, model_pegasus, tokenizer,
            batch_size=2, column_text="dialogue", column_summary="summary"
        )

        rouge_dict = {rn: round(score[rn], 4) for rn in rouge_names}
        df = pd.DataFrame(rouge_dict, index=["pegasus"])
        df.to_csv(self.config.metric_file_name, index=False)


In [ ]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2025-06-11 11:25:13,606: INFO: common: yaml file: D:\projects\AI-Tex-Summraiser\config\config.yaml loaded successfully]
[2025-06-11 11:25:13,612: INFO: common: yaml file: D:\projects\AI-Tex-Summraiser\params.ymal loaded successfully]
[2025-06-11 11:25:13,614: INFO: common: created directory at: artifacts]
[2025-06-11 11:25:13,615: INFO: common: created directory at: artifacts/model_evaluation]


 20%|██        | 1/5 [01:43<06:54, 103.50s/it]

In [ ]:
from pathlib import Path
print(Path("D:/projects/AI-Tex-Summraiser/artifacts/model_trainer/tokenizer").exists())


True
